In [1]:
# 导入项目所需的核心依赖库
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import torchtext
import datasets
import tqdm
import evaluate

Disabling PyTorch because PyTorch >= 2.1 is required but found 2.0.0
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# 设置随机种子，确保实验结果可复现
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 数据集 Dataset

## 1、添加一个Markdown单元格，在其中解释下方单元格的两行代码。
设置 os.environ['HF_ENDPOINT'] = \'https://hf-mirror.com' ，这样做具体改变了什么？
为什么要设置HF_ENDPOINT=\'https://hf-mirror.com'而非直接使用官方源？
dataset = datasets.load_dataset("bentrevett/multi30k") 这行代码具体完成了什么操作？

In [3]:
# 功能：从本地jsonl文件加载Multi30k英德双语翻译数据集
# 说明：替代在线下载方式，直接读取本地已有的训练/验证/测试数据
import json
from datasets import Dataset, DatasetDict

# 数据集本地存储路径
data_dir = r"C:\Users\TX\Desktop\multi30k"

# 工具函数：读取jsonl格式文件，返回Dataset对象
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # 将每行json字符串解析为字典
            data.append(json.loads(line))
    return Dataset.from_list(data)

# 构建完整数据集，包含训练集、验证集、测试集
dataset = DatasetDict({
    "train": load_jsonl(f"{data_dir}/train.jsonl"),       # 训练集
    "validation": load_jsonl(f"{data_dir}/val.jsonl"),    # 验证集
    "test": load_jsonl(f"{data_dir}/test.jsonl")          # 测试集
})

# 打印加载成功信息和数据集结构
print("✅ 数据集加载成功!")
print(dataset)

✅ 数据集加载成功!
DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})


## 代码解释
### 1. `os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`
- **作用**：修改Hugging Face库的默认数据下载端点，将原本的海外官方源替换为国内镜像源`hf-mirror.com`。
- **原因**：官方源服务器在国外，国内访问速度极慢、容易超时甚至下载失败；使用国内镜像可以大幅提升下载速度，保证数据集稳定加载，解决网络访问问题。

### 2. `dataset = datasets.load_dataset("bentrevett/multi30k")`
- **作用**：调用Hugging Face `datasets`库的`load_dataset`方法，自动下载并加载`multi30k`英-德双语机器翻译数据集。
- **细节**：`multi30k`是经典的机器翻译数据集，包含约3万条英德平行语料，代码会自动完成数据集的下载、缓存、解析，并封装为包含`train`/`validation`/`test`三个子集的`DatasetDict`对象，为后续Seq2Seq翻译模型的训练、验证、测试提供数据。

## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [4]:
# 查看完整数据集的结构信息
# 输出显示train/validation/test三个子集的样本数量和特征（'en'、'de'）
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [5]:
# 将完整数据集拆分为训练集、验证集和测试集，分别赋值给变量
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [6]:
# 查看训练集的第0个样本，了解数据格式
# 每个样本包含'en'（英语句子）和'de'（德语句子）两个字段
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [7]:
# 加载spaCy的英文和德语分词模型
# 用于后续对文本进行分词处理，将句子拆分为单词序列
en_nlp = spacy.load("en_core_web_sm")  # 英文分词模型
de_nlp = spacy.load("de_core_news_sm") # 德语分词模型

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [8]:
# 测试英文分词器效果
# 将句子"What a lovely day it is today!"分词为单词和标点序列
string = "What a lovely day it is today!"
[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


## `tokenize_example` 函数作用详解
### 1. 函数核心功能
该函数是 Seq2Seq 翻译任务的**数据预处理核心函数**，用于对英德双语平行语料进行分词、标准化和格式转换，为模型训练准备符合要求的输入数据。

### 2. 逐行代码逻辑拆解
```python
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    # ① 对英语句子分词：用spaCy英文模型分词，提取token文本，截断超长部分
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    # ② 对德语句子分词：用spaCy德文模型分词，提取token文本，截断超长部分
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    
    # ③ 大小写转换（可选）：如果lower=True，将所有token转为小写，统一文本格式
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    
    # ④ 添加起止符：在序列首尾加入<sos>（句子开始）和<eos>（句子结束）标记，告知模型序列边界
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    
    # ⑤ 返回格式化结果：以字典形式返回分词后的英德token序列
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

In [9]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    """
    功能：对单个英德双语样本进行分词和预处理，为模型训练准备数据
    步骤：
    1. 对英语和德语句子分词，截断超过最大长度的部分
    2. 可选：将所有token转为小写，统一文本格式
    3. 在序列首尾添加<sos>（句子开始）和<eos>（句子结束）标记
    4. 返回处理后的英德token序列字典
    """
    # 对英语句子分词，提取token文本并截断超长部分
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    # 对德语句子分词，提取token文本并截断超长部分
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    
    # 若lower=True，将所有token转为小写，统一文本格式
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    
    # 在序列首尾添加<sos>（句子开始）和<eos>（句子结束）标记
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    
    # 返回格式化结果，包含处理后的英德token序列
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


In [10]:
# 设置预处理参数
max_length = 1_000  # 序列最大长度，超过部分将被截断
lower = True        # 是否将所有token转为小写，统一文本格式
sos_token = "<sos>" # 句子开始标记，标识序列起始位置
eos_token = "<eos>" # 句子结束标记，标识序列结束位置

# 将所有参数打包成字典，方便批量传入map函数
fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

# 对训练集、验证集、测试集批量执行预处理函数
train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)  # 处理训练集
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)  # 处理验证集
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)    # 处理测试集

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [11]:
# 验证预处理结果
# 功能：查看训练集第0个样本的处理效果，确认分词、小写转换和特殊标记添加成功
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

In [12]:
# 构建英德双语词汇表
# 功能：根据训练集的分词结果，为英语和德语分别构建词表，将单词映射为数字索引
# 说明：设置最小词频过滤低频词，并添加特殊标记<unk>、<pad>、<sos>、<eos>

# 词表构建参数
min_freq = 2                # 词频下限，出现次数少于2的单词会被过滤
unk_token = "<unk>"         # 未知词标记，代表词表中未收录的单词
pad_token = "<pad>"         # 填充标记，用于将序列补全为相同长度

# 定义需要加入词表的特殊标记
special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

# 构建英语词汇表
en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],  # 训练集的英语分词结果
    min_freq=min_freq,        # 词频过滤
    specials=special_tokens,  # 加入特殊标记
)

# 构建德语词汇表
de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],  # 训练集的德语分词结果
    min_freq=min_freq,        # 词频过滤
    specials=special_tokens,  # 加入特殊标记
)

## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [13]:
# 查看英语词汇表的前10个token
# 功能：验证词表构建是否成功，确认特殊标记和高频词已正确收录
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [14]:
# 查看德语词汇表的前10个token
# 功能：验证词表构建是否成功，确认特殊标记和高频词已正确收录
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [15]:
# 演示get_stoi方法：获取指定单词在英语词表中的索引
# 功能：展示词表的“字符串→数字”映射功能，这里查询单词"the"的索引
en_vocab["the"]

7

In [16]:
# 验证特殊标记的一致性，并获取特殊标记的索引
# 功能：确保英德语词表中<unk>和<pad>标记的索引一致，方便后续处理
assert en_vocab[unk_token] == de_vocab[unk_token]  # 未知词索引一致性检查
assert en_vocab[pad_token] == de_vocab[pad_token]  # 填充词索引一致性检查

unk_index = en_vocab[unk_token]  # 获取<unk>标记的索引
pad_index = en_vocab[pad_token]  # 获取<pad>标记的索引

In [17]:
# 设置词表的默认索引为<unk>标记的索引
# 功能：当遇到词表中未收录的单词时，自动映射为<unk>索引，避免程序报错
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [18]:
# 演示lookup_indices方法：将token列表转换为索引列表
# 功能：展示词表的批量“字符串→数字”映射功能，为后续模型输入做准备
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [19]:
# 演示lookup_tokens方法：将索引列表转换回token列表
# 功能：展示词表的反向映射功能，将模型输出的数字序列还原为单词序列
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


In [20]:
def numericalize_example(example, en_vocab, de_vocab):
    """
    功能：将分词后的token列表转换为数字索引序列，完成文本向量化
    步骤：
    1. 使用英语词表，将英文token列表映射为数字索引（en_ids）
    2. 使用德语词表，将德文token列表映射为数字索引（de_ids）
    3. 返回包含数字序列的字典，为模型训练提供输入数据
    """
    en_ids = en_vocab.lookup_indices(example["en_tokens"])  # 英文token转数字索引
    de_ids = de_vocab.lookup_indices(example["de_tokens"])  # 德文token转数字索引
    return {"en_ids": en_ids, "de_ids": de_ids}

In [21]:
# 打包词表参数，方便map函数调用
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

# 对训练集、验证集、测试集批量执行数值化处理
train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)  # 处理训练集
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)  # 处理验证集
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)    # 处理测试集

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [22]:
# 查看训练集第0个样本的最终处理结果
# 功能：验证'en_ids'和'de_ids'字段已成功添加，确认文本已转换为数字索引序列
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [23]:
# 将数据格式转换为PyTorch张量，为模型训练做准备
# 功能：将en_ids和de_ids字段转换为torch.Tensor格式，同时保留所有原始特征
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

# 对训练集、验证集、测试集统一设置格式
train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)
valid_data = valid_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)
test_data = test_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [24]:
# 验证数据格式转换结果
# 功能：检查训练集第0个样本，确认'en_ids'和'de_ids'已成功转换为PyTorch张量
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

In [25]:
def get_collate_fn(pad_index):
    """
    功能：创建一个批次数据整理函数，用于将不同长度的序列填充为相同长度
    说明：解决模型训练中批次内序列长度不一致的问题
    """
    def collate_fn(batch):
        # 提取批次中所有样本的英文和德文序列
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        
        # 使用pad_sequence将序列填充为批次内的最大长度，填充值为pad_index
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        
        # 返回整理好的批次数据字典
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [26]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    """
    功能：创建数据加载器，封装批次整理、数据打乱和迭代逻辑
    参数：
        dataset: 已预处理好的数据集
        batch_size: 批次大小
        pad_index: 填充标记的索引
        shuffle: 是否打乱数据（训练集需打乱，验证/测试集不需要）
    """
    collate_fn = get_collate_fn(pad_index)  # 获取批次整理函数
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [27]:
# 设置批次大小
batch_size = 128

# 创建训练集、验证集、测试集的数据加载器
train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

In [28]:
class Encoder(nn.Module):
    """
    Seq2Seq模型的编码器
    功能：将源语言（英语）序列编码为上下文向量，为解码器提供信息
    """
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim    # LSTM隐藏层维度
        self.n_layers = n_layers        # LSTM层数
        self.embedding = nn.Embedding(input_dim, embedding_dim)  # 词嵌入层
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)  # LSTM层
        self.dropout = nn.Dropout(dropout)  # Dropout层，防止过拟合

    def forward(self, src):
        # src形状: [src_length, batch_size]
        embedded = self.dropout(self.embedding(src))  # 词嵌入+Dropout
        # embedded形状: [src_length, batch_size, embedding_dim]
        outputs, (hidden, cell) = self.rnn(embedded)  # LSTM前向传播
        # outputs形状: [src_length, batch_size, hidden_dim * n_directions]
        # hidden形状: [n_layers * n_directions, batch_size, hidden_dim]
        # cell形状: [n_layers * n_directions, batch_size, hidden_dim]
        return hidden, cell  # 返回编码器的最终隐藏状态和细胞状态

# 解码器 Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

In [29]:
class Decoder(nn.Module):
    """
    Seq2Seq模型的解码器
    功能：接收编码器的上下文向量，逐步生成目标语言序列
    """
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim     # 目标语言词表大小
        self.hidden_dim = hidden_dim     # LSTM隐藏层维度
        self.n_layers = n_layers         # LSTM层数
        self.embedding = nn.Embedding(output_dim, embedding_dim)  # 词嵌入层
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)  # LSTM层
        self.fc_out = nn.Linear(hidden_dim, output_dim)  # 全连接层，输出词表概率
        self.dropout = nn.Dropout(dropout)  # Dropout层，防止过拟合

    def forward(self, input, hidden, cell):
        # input: [batch_size]
        # hidden: [n_layers * n_directions, batch_size, hidden_dim]
        # cell: [n_layers * n_directions, batch_size, hidden_dim]
        # 解码器是单向的，n_directions=1，因此hidden/cell形状变为[n_layers, batch_size, hidden_dim]

        input = input.unsqueeze(0)  # 增加序列维度，形状变为[1, batch_size]
        embedded = self.dropout(self.embedding(input))  # 词嵌入+Dropout
        # embedded: [1, batch_size, embedding_dim]

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))  # LSTM前向传播
        # output: [1, batch_size, hidden_dim]
        # hidden: [n_layers, batch_size, hidden_dim]
        # cell: [n_layers, batch_size, hidden_dim]

        prediction = self.fc_out(output.squeeze(0))  # 全连接层输出预测
        # prediction: [batch_size, output_dim]

        return prediction, hidden, cell

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

### Seq2Seq模型说明
#### 模型功能
整合Encoder和Decoder，实现端到端的机器翻译，输入源语言序列，输出目标语言序列。

#### forward函数流程
1.  编码器处理：源语言序列输入Encoder，得到上下文向量（隐藏状态和细胞状态）。
2.  解码器初始化：以`<sos>`标记作为第一个输入，结合编码器的上下文向量，开始逐步解码。
3.  循环生成序列：在循环中，解码器根据前一步的输出，生成下一个词的预测。
4.  Teacher Forcing机制：在每一步解码时，按设定概率选择使用真实标签或模型预测词作为下一步输入，避免错误累积。

In [30]:
class Seq2Seq(nn.Module):
    """
    整合Encoder和Decoder的Seq2Seq模型，实现端到端机器翻译
    """
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder  # 编码器
        self.decoder = decoder  # 解码器
        self.device = device    # 设备信息（CPU/GPU）

        # 检查编码器和解码器的隐藏层维度、层数是否一致
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src: [src_length, batch_size]
        # trg: [trg_length, batch_size]
        # teacher_forcing_ratio: 使用真实标签作为输入的概率

        batch_size = trg.shape[1]       # 批次大小
        trg_length = trg.shape[0]       # 目标序列长度
        trg_vocab_size = self.decoder.output_dim  # 目标词表大小

        # 初始化输出张量，用于存储解码器的预测结果
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)

        # 编码器前向传播，获取上下文向量
        hidden, cell = self.encoder(src)
        # hidden: [n_layers * n_directions, batch_size, hidden_dim]
        # cell: [n_layers * n_directions, batch_size, hidden_dim]

        # 解码器第一个输入是目标序列的第一个词，即<sos>标记
        input = trg[0, :]

        # 循环生成目标序列（从第1个词到第trg_length-1个词）
        for t in range(1, trg_length):
            # 解码器前向传播，获取当前时间步的预测和新的状态
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output: [batch_size, output_dim]
            # hidden: [n_layers, batch_size, hidden_dim]
            # cell: [n_layers, batch_size, hidden_dim]

            # 保存当前时间步的预测结果
            outputs[t] = output

            # 决定是否使用Teacher Forcing
            teacher_force = random.random() < teacher_forcing_ratio

            # 获取当前预测的最高概率词
            top1 = output.argmax(1)

            # 下一步的输入：使用真实标签（teacher force）或模型预测词
            input = trg[t] if teacher_force else top1

        return outputs

# 模型训练

模型初始化

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [31]:
# 定义模型超参数
input_dim = len(de_vocab)        # 源语言（德语）词表大小
output_dim = len(en_vocab)       # 目标语言（英语）词表大小
encoder_embedding_dim = 256      # 编码器词嵌入维度
decoder_embedding_dim = 256      # 解码器词嵌入维度
hidden_dim = 512                 # LSTM隐藏层维度
n_layers = 2                      # LSTM层数
encoder_dropout = 0.5            # 编码器Dropout率
decoder_dropout = 0.5            # 解码器Dropout率
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 设备选择（GPU/CPU）

# 编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

# 解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

# Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)

权重初始化

In [32]:
# 定义权重初始化函数：对模型所有参数进行均匀分布初始化
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)  # 均匀分布初始化，范围[-0.08, 0.08]

# 对模型应用权重初始化
model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [33]:
def count_parameters(model):
    # 统计模型中所有可训练参数的数量
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 打印模型可训练参数总数
print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


优化器 optimizer

In [34]:
# 使用Adam优化器，传入模型参数
optimizer = optim.Adam(model.parameters())

损失函数 Loss Function

In [35]:
# 定义交叉熵损失函数，忽略填充标记的索引
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [36]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()  # 设置模型为训练模式（启用Dropout等训练层）
    epoch_loss = 0  # 初始化当前epoch的损失

    # 遍历训练集的每个批次
    for i, batch in enumerate(data_loader):
        src = batch["de_ids"].to(device)  # 获取源语言序列并移动到指定设备
        trg = batch["en_ids"].to(device)  # 获取目标语言序列并移动到指定设备
        # src: [src_length, batch_size]
        # trg: [trg_length, batch_size]

        optimizer.zero_grad()  # 梯度清零，避免累积

        # 模型前向传播，得到预测结果
        output = model(src, trg, teacher_forcing_ratio)
        # output: [trg_length, batch_size, trg_vocab_size]

        output_dim = output.shape[-1]  # 获取目标词表大小
        # 去掉序列的第一个元素（<sos>标记），并展平为二维
        output = output[1:].view(-1, output_dim)
        # output: [(trg_length - 1) * batch_size, trg_vocab_size]

        # 目标序列同样去掉第一个元素，并展平为一维
        trg = trg[1:].view(-1)
        # trg: [(trg_length - 1) * batch_size]

        loss = criterion(output, trg)  # 计算交叉熵损失
        loss.backward()  # 反向传播计算梯度

        # 梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()  # 更新模型参数

        epoch_loss += loss.item()  # 累加当前批次的损失

    # 返回当前epoch的平均损失
    return epoch_loss / len(data_loader)

Evaluation Loop:

In [37]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()  # 设置模型为评估模式（禁用Dropout等训练层）
    epoch_loss = 0  # 初始化当前epoch的损失

    with torch.no_grad():  # 禁用梯度计算，节省内存和计算资源
        # 遍历验证集的每个批次
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)  # 获取源语言序列并移动到指定设备
            trg = batch["en_ids"].to(device)  # 获取目标语言序列并移动到指定设备
            # src: [src_length, batch_size]
            # trg: [trg_length, batch_size]

            # 模型前向传播，评估时关闭Teacher Forcing
            output = model(src, trg, 0)
            # output: [trg_length, batch_size, trg_vocab_size]

            output_dim = output.shape[-1]  # 获取目标词表大小
            # 去掉序列的第一个元素（<sos>标记），并展平为二维
            output = output[1:].view(-1, output_dim)
            # output: [(trg_length - 1) * batch_size, trg_vocab_size]

            # 目标序列同样去掉第一个元素，并展平为一维
            trg = trg[1:].view(-1)
            # trg: [(trg_length - 1) * batch_size]

            loss = criterion(output, trg)  # 计算交叉熵损失
            epoch_loss += loss.item()  # 累加当前批次的损失

    # 返回当前epoch的平均损失
    return epoch_loss / len(data_loader)

# 模型训练

In [38]:
n_epochs = 3  # 训练轮数，提升模型泛化能力与翻译质量
clip = 1.0    # 梯度裁剪阈值，防止梯度爆炸
teacher_forcing_ratio = 0.5  # Teacher Forcing概率

best_valid_loss = float("inf")  # 初始化最佳验证损失

# 遍历每个训练轮次
for epoch in tqdm.tqdm(range(n_epochs)):
    # 训练模型并获取训练损失
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )

    # 在验证集上评估模型性能并获取验证损失
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )

    # 如果当前验证损失更低，则更新最佳模型
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")  # 保存模型参数

    # 打印训练和验证结果，计算并输出困惑度（PPL）
    print(f"\tTrain Loss: {train_loss:.3f} | Train PPL: {np.exp(train_loss):.3f}")
    print(f"\tValid Loss: {valid_loss:.3f} | Valid PPL: {np.exp(valid_loss):.3f}")

 33%|███████████████████████████▋                                                       | 1/3 [06:41<13:22, 401.40s/it]

	Train Loss: 5.026 | Train PPL: 152.323
	Valid Loss: 4.838 | Valid PPL: 126.254


 67%|███████████████████████████████████████████████████████▎                           | 2/3 [13:26<06:43, 403.58s/it]

	Train Loss: 4.396 | Train PPL: 81.143
	Valid Loss: 4.785 | Valid PPL: 119.741


100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [20:12<00:00, 404.32s/it]

	Train Loss: 4.150 | Train PPL: 63.437
	Valid Loss: 4.536 | Valid PPL: 93.282


# 模型验证

In [39]:
# 加载训练好的模型参数
model.load_state_dict(torch.load("tut1-model.pt"))

<All keys matched successfully>

In [40]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    """
    翻译句子的函数
    输入：德语句子/分词列表
    输出：英语翻译结果的token列表
    """
    model.eval()  # 设置模型为评估模式
    with torch.no_grad():  # 禁用梯度计算
        # 如果输入是字符串，先进行分词
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]

        # 可选：将所有token转为小写
        if lower:
            tokens = [token.lower() for token in tokens]

        # 添加句子开始和结束标记
        tokens = [sos_token] + tokens + [eos_token]

        # 将token列表转换为数字索引
        ids = de_vocab.lookup_indices(tokens)

        # 转换为PyTorch张量，并移动到指定设备
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)

        # 编码器前向传播，获取上下文向量
        hidden, cell = model.encoder(tensor)

        # 解码器的第一个输入是<sos>标记
        inputs = en_vocab.lookup_indices([sos_token])

        # 循环生成目标序列
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)

            # 获取当前时间步预测的最高概率词
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)

            # 如果预测到<eos>标记，提前结束循环
            if predicted_token == en_vocab[eos_token]:
                break

        # 将数字索引转换回token列表
        tokens = en_vocab.lookup_tokens(inputs)

    return tokens

In [41]:
# 获取测试集第0个样本的源语言句子和目标翻译
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [42]:
# 22、测试模型翻译效果
# 功能：调用训练好的模型，对测试集的样本进行翻译，验证模型性能
# 说明：由于训练轮数（epoch）仅为1轮，模型尚未充分收敛，翻译效果有限
#      可通过增加训练轮数、调整超参数，进一步观察loss变化与翻译质量提升

translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

translation

['<sos>',
 'a',
 'man',
 'in',
 'a',
 'blue',
 'shirt',
 'is',
 'a',
 'a',
 'a',
 '.',
 '.',
 '<eos>']

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [43]:
translation

['<sos>',
 'a',
 'man',
 'in',
 'a',
 'blue',
 'shirt',
 'is',
 'a',
 'a',
 'a',
 '.',
 '.',
 '<eos>']